# R/S benchmark — 1. Dataset generation

Stage 1 of the split benchmark pipeline. This notebook **only** runs the emulator and saves the
resulting lambda datasets — no PCE is fitted here. Stage 2, in
[`02_train_pce.ipynb`](02_train_pce.ipynb), only reads what this notebook writes.

**State limit function**

$$g = k(t) \cdot \frac{R}{z_1} - S \cdot z_2$$

$z_1$ and $z_2$ are normal latent multipliers (mean 1.0, sd 0.028 and 0.096); $k(t) = 1 +
(k_{final}-1)\,t/100$ is the degradation factor. Full context and the original single-notebook
version of this pipeline are in [`../pipeline_benchmark.ipynb`](../pipeline_benchmark.ipynb).

Functions come from [`functions_final.py`](../functions_final.py), one directory up:
`generate_dataset_at_time_benchmark` calls `emulator_function_time_benchmark` twice per time step
(once on the training design points, once on a fresh validation sample) and saves both.

**Artefacts written per time step**, `<n_latent_samples>_<kind>_<split>_<t>_benchmark.pkl`:

| kind | split | content |
|---|---|---|
| `dataset_full` | train / val | one row per latent replica: $R$, $S$, $z_1$, $z_2$, $g$, lambdas, processing time |
| `dataset_unique` | train / val | one row per design point: $R$, $S$, the four lambdas, processing time |

Plus one aggregate `<n_latent_samples>_emulator_timing_benchmark.pkl`, read back by stage 2 to
compute the emulator/surrogate speed-up.

## 1. Libraries

In [ ]:
import sys
import time
from pathlib import Path

# functions_final.py sits one directory up, in beam_problem_1/
sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd

from functions_final import *
from UQpy.distributions import Normal, JointIndependent

## 2. Random variables and fixed parameters

Design variables $R$ and $S$, plus everything the emulator needs that isn't a design variable.

In [ ]:
r_mean = 5.0   # resistance mean
r_std  = 0.8   # resistance standard deviation
s_mean = 2.0   # load mean
s_std  = 0.6   # load standard deviation

cement_type_note = None  # not applicable to the benchmark — kept out on purpose

n_samples            = 1000     # Number of design samples
n_latent_samples     = 100000   # Number of latent samples per design sample. Also the filename prefix
n_samples_validation = 250      # Number of validation samples, redrawn at every time step
n_lambdas            = 4        # Number of λs (λ1, λ2, λ3, λ4)
k_factor_final        = 0.3      # Degradation factor at t = 100. Use 1.0 for no time effect
z1_std                = 0.028    # Standard deviation of the resistance latent multiplier
z2_std                = 0.096    # Standard deviation of the load latent multiplier

## 3. Design samples

In [ ]:
r_dist = Normal(loc=r_mean, scale=r_std)
s_dist = Normal(loc=s_mean, scale=s_std)
joint  = JointIndependent(marginals=[r_dist, s_dist])

x_pce_rvs = joint.rvs(n_samples)

print("Samples generated successfully!")
print(f"   Number of design samples: {n_samples}")
print(f"   Number of latent samples per design sample: {n_latent_samples}")
print(f"   Total simulations per time step: {(n_samples + n_samples_validation) * n_latent_samples}")

## 4. Time grid

In [ ]:
times = np.linspace(0, 150, 10, endpoint=True)  # Time points for the degradation factor
# times = [10, 20, 30, 40]
times

## 5. Generate the dataset at each time step

A fresh validation sample is drawn for every time step (matching the original combined pipeline).
`generate_dataset_at_time_benchmark` does the rest: latent sampling, $g$ evaluation, GLD fit, and
saving `dataset_full`/`dataset_unique` for both splits.

In [ ]:
print("="*60)
print("GENERATING THE BENCHMARK DATASET")
print("="*60)

generation_results = []
for t in times:
    x_val = joint.rvs(n_samples_validation)
    result = generate_dataset_at_time_benchmark(
                                                   x_train=x_pce_rvs,
                                                   x_val=x_val,
                                                   time_step=t,
                                                   n_latent_samples=n_latent_samples,
                                                   k_factor_final=k_factor_final,
                                                   z1_std=z1_std,
                                                   z2_std=z2_std,
                                                   output_dir='.',
                                               )
    generation_results.append(result)

## 6. Timing summary

Cost of building the dataset, per time step. `Train total (s)` is what stage 2 will compare against
the PCE's own evaluation time to compute the speed-up.

In [ ]:
timing_rows = []
for result in generation_results:
    train_t = result['df_unique_train']['Processing time (s)']
    val_t   = result['df_unique_val']['Processing time (s)']
    timing_rows.append({
                           'Time (years)':   result['time_step'],
                           'n_train':        len(train_t),
                           'Train total (s)': train_t.sum(),
                           'Train mean (ms)': train_t.mean() * 1e3,
                           'n_val':          len(val_t),
                           'Val total (s)':  val_t.sum(),
                       })

emulator_timing = pd.DataFrame(timing_rows)

with open(f'{n_latent_samples}_emulator_timing_benchmark.pkl', 'wb') as f:
    dill.dump(emulator_timing, f)

print(f"Total emulator time over the whole grid: {emulator_timing['Train total (s)'].sum() + emulator_timing['Val total (s)'].sum():.1f} s")
emulator_timing